In [1]:
# Import Libraries
import pandas as pd
import numpy as np

# Text preprocessing
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Evaluation
from sklearn.metrics import accuracy_score, classification_report

I0000 00:00:1778267892.862205   34394 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778267892.863166   34394 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778267893.133481   34394 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778267894.487028   34394 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [3]:
# ==========================================
# 1. Load Dataset
# ==========================================

# Load CSV file
df = pd.read_csv("IMDB Dataset.csv")

# Display first 5 rows
print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [4]:
# ==========================================
# 2. Prepare Input and Output
# ==========================================

# Input reviews
X = df['review']

# Output labels
# positive -> 1
# negative -> 0

y = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})


In [5]:

# ==========================================
# 3. Convert Text into Numbers
#    using TF-IDF Vectorization
# ==========================================

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X = vectorizer.fit_transform(X).toarray()


In [6]:

# ==========================================
# 4. Train-Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ===

In [7]:
# ==========================================
# 5. Build Deep Neural Network
# ==========================================

model = Sequential()

# Input + Hidden Layer
model.add(Dense(128, activation='relu',input_shape=(5000,)))

# Dropout layer helps reduce overfitting
model.add(Dropout(0.3))

# Hidden Layer
model.add(Dense(64, activation='relu'))

# Another Dropout layer
model.add(Dropout(0.3))

# Output Layer
# sigmoid activation for binary classification
model.add(Dense(1, activation='sigmoid'))


/home/darshan/.local/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1778267968.414339   34394 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
# ==========================================
# 6. Compile Model
# ==========================================

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ==========================================

In [9]:

# ==========================================
# 7. Train Model
# ==========================================

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)


Epoch 1/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8559 - loss: 0.3421 - val_accuracy: 0.8880 - val_loss: 0.2713
Epoch 2/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9081 - loss: 0.2311 - val_accuracy: 0.8865 - val_loss: 0.2794
Epoch 3/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9327 - loss: 0.1791 - val_accuracy: 0.8767 - val_loss: 0.3041
Epoch 4/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9612 - loss: 0.1164 - val_accuracy: 0.8775 - val_loss: 0.3512
Epoch 5/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9818 - loss: 0.0609 - val_accuracy: 0.8770 - val_loss: 0.4340


In [10]:

# ==========================================
# 8. Evaluate Model
# ==========================================

loss, accuracy = model.evaluate(X_test, y_test)

print("\nTest Accuracy:", accuracy)


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8747 - loss: 0.4399

Test Accuracy: 0.8747000098228455


In [11]:
# ==========================================
# 9. Predictions
# ==========================================

y_pred = model.predict(X_test)

# Convert probabilities into 0 or 1
y_pred = (y_pred > 0.5).astype("int32")


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [12]:
# ==========================================
# 10. Classification Report
# ==========================================

print("\nClassification Report:\n")

print(classification_report(y_test, y_pred))


Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.87      0.87      4961
           1       0.87      0.88      0.88      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



In [13]:
# ==========================================
# 11. Test with Custom Review
# ==========================================

sample_review = [
    "The movie was fantastic and acting was amazing"
]

# Convert text into TF-IDF features
sample_vector = vectorizer.transform(sample_review).toarray()

# Predict sentiment
prediction = model.predict(sample_vector)

if prediction[0][0] > 0.5:
    print("\nPositive Review 😊")
else:
    print("\nNegative Review 😞")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step

Positive Review 😊
